In [21]:
!pip install pypdf langchain langchain-aws langchain-community faiss-cpu -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.5/182.5 kB 12.8 MB/s eta 0:00:00


In [1]:
from google.colab import files
uploaded = files.upload()

Saving budget_speech.pdf to budget_speech.pdf


In [23]:
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_REGION'] = userdata.get('AWS_REGION')


# Load PDF Documents

In [53]:
# Import Library
from langchain_community.document_loaders import PyPDFLoader


# Load PDF Document
def read_doc(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents


# Load PDF
doc = read_doc("budget_speech.pdf")

In [54]:
# Check loaded documents
print("Total pages:", len(doc))

# Print first page
print(doc[0].page_content)

Total pages: 58
GOVERNMENT OF INDIA
BUDGET 2023-2024
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2023


# Split Documents into Chunks

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_data(docs, chunk_size=800, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    documents = text_splitter.split_documents(docs)

    return documents


# Create chunks
documents = chunk_data(doc)

In [57]:
print("Total pages:", len(doc))
print("Total chunks:", len(documents))
print(documents[0].page_content)

Total pages: 58
Total chunks: 140
GOVERNMENT OF INDIA
BUDGET 2023-2024
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2023


# Initialize Bedrock Embeddings

In [59]:
from langchain_aws import BedrockEmbeddings

# Create Bedrock Embeddings
embeddings = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0"
)
embeddings

BedrockEmbeddings(client=<botocore.client.BedrockRuntime object at 0x7d3003a106e0>, region_name=None, credentials_profile_name=None, aws_access_key_id=SecretStr('**********'), aws_secret_access_key=SecretStr('**********'), aws_session_token=None, model_id='amazon.titan-embed-text-v2:0', model_kwargs=None, provider=None, endpoint_url=None, normalize=False, dimensions=None, config=None)

In [61]:
vector = embeddings.embed_query("How are you")

print("Embedding dimensions:", len(vector))

Embedding dimensions: 1024


# Create FAISS Vector Store

In [62]:
from langchain_community.vectorstores import FAISS

# Create FAISS vector database
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


# Create Retriever

In [66]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# Create Bedrock LLM for generation

In [64]:
from langchain_aws import ChatBedrockConverse

# Create Bedrock LLM
llm = ChatBedrockConverse(
    model="amazon.nova-pro-v1:0",
    region_name=os.environ["AWS_REGION"],
    temperature=0.5
)

# Create Chat Prompt

In [65]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question using only the information provided in the context.

If the answer is not available in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
""")

# Format retrieved documents

In [67]:
# Format retrieved documents
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )


# Create RAG Chain

In [68]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG Chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test RAG Pipeline

In [69]:
query = "How much the agriculture target will be increased by how many crore?"

answer = rag_chain.invoke(query)

print(answer)

The agriculture credit target will be increased to ₹20 lakh crore. 

To break it down:
- "lakh" means 100,000.
- Therefore, ₹20 lakh crore is equivalent to ₹20,00,000 crore.

However, the context does not specify the current agriculture credit target, so I can't determine by how many crore it will be increased. 

So, based on the provided document, I don't know by how many crore the agriculture credit target will be increased.
